> **Standalone pipeline（2026-10-18）**：本 Notebook 的代码现由本目录的
> `pipeline/` 提供，无需 `../Jev-Mem` 克隆。默认 `JEV_MEM_MODE=offline`：
> 哈希词汇向量、启发式决策、抽取式回答、lexical F1，仅验证执行流程。
> 真 API 请设置 `JEV_MEM_MODE=live`，详见 [PIPELINE.md](PIPELINE.md)。
> 下文论文介绍、原上游路径、原始实验表格与解读保留为历史研究笔记，
> 不表示本实现的新实测结果。所有旧代码输出已清除；新结果以 run 的 summary.json 为准。

# Jev-Mem 走读与实验：给 Agent 装一个"会自己整理的记忆"

**TL;DR**：Jev-Mem 是 UT Dallas 2026 年 9 月 21 日开源的 Agent 记忆架构（[arXiv 2609.23986](https://arxiv.org/abs/2609.23986)，第二天登上 [Hugging Face 趋势榜](https://huggingface.co/papers/2609.23986)）。它的核心主张只有一句话：**记忆管理里那些高频小决策（该不该存、和谁有关、去哪找、找到没够），不值得动用大模型——交给 Jev 这样的轻量 System-One 决策模型；大模型只在最后组织答案时出场（System-Two）**。论文在 LoCoMo 基准上报告：答案质量比最强基线高 11%，建记忆快 6.6 倍，查询延迟低 36.7%。

这个 notebook 做三件事：

1. **讲明白**：Jev-Mem 的三个平面架构，以及"写入"和"读取"两条管线里 Jev 到底在做什么决策（对着真实代码走）；
2. **跑起来**：用一段中文多轮对话建记忆、做检索，System-Two 用 **DeepSeek**（`deepseek-chat`），Jev 决策有 `TYPESAFE_API_KEY` 就用真模型，没有就走代码兜底（每一步都会如实标注）；
3. **比一比**：同一组记忆问答，对比 4 种做法——滑动窗口、全上下文、朴素向量 RAG（= 卸掉 Jev 决策面）、Jev-Mem 完整管线，看质量、延迟、token 花销。

> 本地文件：代码仓库 `../Jev-Mem/`，论文 PDF `../Jev-Mem/docs/jev_mem_paper_2609.23986.pdf`。
> 运行前把 key 写进 `../Jev-Mem/.env`（格式见下面第 1 节）。没有 key 也能跑通大部分内容。

> **续篇**：[jev_mem_scaling.ipynb](jev_mem_scaling.ipynb) 把同一组四臂实验拉到 48~384 轮的长程场景，并附真 LoCoMo（419 轮）对照——长对话下质量反转与 token 剪刀差都在那一篇。末节有三组实验的结论总览与选型判据。

## 1. 背景：Jev 一個模型不够吗？

在前面的 cookbook 章节里，Jev 的标准用法是"一次决策调用"：发一段状态 + 一组类型化问题（Noul 命题概率 / Choice 选项分布），拿回结构化答案。这在单点任务上很好用，但**长期运行的 Agent 记忆**是一整条流水线，每一步都塞着小决策：

| 记忆环节 | 里面藏着的决策 | 用大模型做的问题 |
|---|---|---|
| 写入 | 这句话值得存吗？是情节、语义、程序还是偏好？ | 每条记忆一次生成调用，慢且贵 |
| 建联 | 新记忆和哪条旧记忆有语义/因果/实体关系？ | 候选两两比较，调用量随记忆数增长 |
| 检索 | 该查哪几张关系图？走多深？这条候选有用吗？ | 检索路径上反复生成，延迟被放大 |
| 停止 | 证据够了吗？再找一轮值不值？ | 不停就超时，停早了漏证据 |

A-MEM、MemoryOS 这类系统把上述决策全交给大模型自由生成，论文实测它们建一次记忆要 3000 秒以上。Jev-Mem 的做法是把这条决策链**整体搬到一个专门的 System-One 控制面**上，用类型化、带概率、可批量的小决策替代自由生成。

**论文信息**

| 项目 | 内容 |
|---|---|
| 标题 | Jev-Mem: System-One-Controlled Agentic Memory for Efficient AI Agents |
| 作者 | Dongming Jiang, Yi Li, Bingzhe Li（UT Dallas） |
| 时间 | 2026-09-21 开源，09-22 登上 HF Papers 趋势榜 |
| 代码 | https://github.com/libingzheren/Jev-Mem |
| 论文 | https://arxiv.org/abs/2609.23986 |

## 2. 架构：三个平面，两条管线

![Jev-Mem 架构](overall_structure.png)

三个平面各管一件事：

| 平面 | 谁来做 | 职责 |
|---|---|---|
| **System-One 控制面** | Jev（`jev-latest`，走 TypeSafe API） | 高频、结构化的小决策：打类型分、判关系、选路由、给预算、评候选、判停止 |
| **记忆面** | 普通代码（networkx + 向量库 + BM25） | 保存原始观察（带时间戳和出处），维护 semantic / temporal / causal / entity 四张关系图视图，向量+关键词双索引 |
| **System-Two 推理面** | 任意 OpenAI 兼容大模型（**本 notebook 用 DeepSeek**） | 只做两件事：答复杂问题、组织最终答案 |

两条管线的决策点（对着代码）：

**写入**（`memory/memory_builder.py: build()`）：
观察进来 → ① Jev 判类型（episodic/semantic/procedural/preference 四个独立概率）→ ② 代码用向量+关键词+实体+时间找出有界候选集（默认 top-10）→ ③ Jev 逐对判关系（语义相关？谁导致谁？同一实体？），时间先后这类确定性关系由代码直接建 → ④ 入图，四张视图同步更新。默认档案**不删任何原始观察**——现在没用的细节以后可能有用。

**读取**（`memory/jev_mem_retrieval.py: RetrievalController.query()`）：
问题进来 → ① Jev 路由：四张关系图各需要多少（Noul 概率），要不要多跳，要不要最新事实 → ② 向量+关键词锚点起步 → ③ 代码按路由概率分配**整数遍历预算**（最大余数法，总数不超上限）→ ④ 沿图扩展，Jev 给每个新候选打分（有用/新信息/佐证）→ ⑤ 每轮结束 Jev 判停止：证据够了吗？再找还有用吗？→ ⑥ 选出的证据交给 System-Two 组织答案。

**Jev 一共被问了 7 组问题**（定义在 `memory/jev_questions.py`，全部是 Noul 概率，只有整理决策用一个 Choice）：

| 问题组 | 时机 | 内容 |
|---|---|---|
| admission | 写入 | 值得存吗 / 未来有用吗 / 重要吗 / 新颖吗 / 冗余吗 |
| memory_type | 写入 | episodic / semantic / procedural / preference 四个独立类型分 |
| relation | 写入 | 和每条候选：语义相关？谁导致谁？同一实体？ |
| consolidation | 写入（周期） | 重复/矛盾/过时/该合并还是该抽象（唯一 Choice） |
| routing | 读取 | 四张图各需要吗、要多跳吗、要最新吗 |
| traversal | 读取 | 这条候选相关吗 / 关系有用吗 / 有新信息吗 / 佐证吗 |
| stopping | 读取 | 证据够吗 / 再找有用吗 / 缺什么 / 有矛盾吗 |

## 3. 环境准备

**一次性准备**（本机已做好）：

```bash
cd ../Jev-Mem
python3.11 -m venv .venv && source .venv/bin/activate
pip install -r requirements.txt
```

**配置 key**：编辑 `../Jev-Mem/.env`（没有就新建）：

```dotenv
# System-Two：DeepSeek（OpenAI 兼容接口，代码零改动）
DEEPSEEK_API_KEY=sk-你的deepseek-key
# System-One：真 Jev 决策（可选，没有则走代码兜底并在输出中标注）
TYPESAFE_API_KEY=你的typesafe-key
```

- 有 DeepSeek key：第 5、6 节实验可跑。
- 有 Typesafe key：Jev-Mem 臂用真 Jev 决策；没有时用确定性兜底逻辑，**结果会打折，但管线完整**，输出里会明确标注决策来源（jev / fallback）。
- embedding 走硅基流动的 `Qwen/Qwen3-Embedding-0.6B`（OpenAI 兼容 `/embeddings`，1024 维，中文强）：在 `.env` 加 `SILICONFLOW_API_KEY=sk-...`。没有该 key 时自动回落本地 `paraphrase-multilingual-MiniLM-L12-v2`（首次从 HuggingFace 下载），管线其余部分不变。

In [ ]:
import os, sys, json
from pathlib import Path

# 本地 standalone 包；无需上游克隆，也不改变工作目录。
cwd = Path.cwd().resolve()
ROOT = next((p for p in (cwd, *cwd.parents)
             if (p / "Jev_mem" / "pipeline" / "memory.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("请从 LLM-Notebooks 根目录或 Jev_mem/ 启动 Notebook")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from Jev_mem.pipeline.notebooks import make_session
from Jev_mem.pipeline.data import walkthrough_dataset, synthetic_dataset
from Jev_mem.pipeline.memory import Memory

MODE = os.getenv("JEV_MEM_MODE", "offline")
OUTPUT = Path(os.getenv("JEV_MEM_OUTPUT", str(ROOT / "Jev_mem" / "artifacts")))
print("Implementation: standalone | Mode:", MODE)


## 4. 离线走读 · 写入管线：一条观察是怎么变成"记忆"的

检测到 `TYPESAFE_API_KEY` 时，这段走读直接用**真 Jev**（审计日志里 `source=jev`）；没有 key 时自动挂上**教学 mock 决策器**——它模拟 Jev 的调用契约（同样的 state + questions → 概率），用中文关键词启发式给分（审计里 `source=mock`）。两种模式的管线一行不改：这正是 Jev-Mem 的可编程性，**决策面是可插拔的**。

In [ ]:
session = make_session(MODE)
encoder = session.encoder
print("embedding:", encoder.fingerprint)
print("decisions:", session.decisions.source)
print("answer backend:", session.answerer.source)


In [ ]:
import tempfile
DATASET = walkthrough_dataset()
CACHE = Path(tempfile.mkdtemp(prefix="jev-mem-walkthrough-"))
memory = Memory(encoder=session.encoder, decisions=session.decisions,
                audit_path=CACHE / "decisions.jsonl")
observations = [o for o in DATASET["observations"] if o["role"] == "user"][:6]
print(memory.build(observations))
for node in memory.nodes:
    print(node["content"][:40], node["memory_type"])


In [ ]:
from collections import Counter
print("Graph views:", dict(Counter(edge["kind"] for edge in memory.edges)))
for edge in memory.edges[:5]:
    print(edge)


**注意两件事**：

1. 每个节点都保存了 `raw_content`（原文）和 `jev_mem` 决策元数据——证据永远不丢，决策可回放；
2. 类型分是**四个独立的 Noul 概率**，不是四选一——"上周体检发现血脂偏高"可以同时是高情节性（一个具体事件）和高语义性（一个持久事实），Jev-Mem 不强迫它们互斥。

## 5. 离线走读 · 读取管线：一个问题是怎么被"路由"到证据的

In [ ]:
for question in ("用户的上海出差为什么改期了？现在定在哪天？", "用户对饮食有什么要注意的？"):
    result = memory.retrieve(question)
    print(question)
    print(result["diagnostics"])
    print(result["context"], "\n")


In [ ]:
print("审计日志最后 3 条：")
for line in (CACHE / "decisions.jsonl").read_text().splitlines()[-3:]:
    print(json.loads(line))


## 6. 实验设计：和"不用 Jev-Mem"比一比

一段 16 轮、跨 4 个日期的中文人设对话（用户 = 提需求的人，助手 = 记忆方），10 个问题覆盖多跳、时间、单跳、对抗四类。**四个臂**：

| 臂 | 做法 | 回答的问题 |
|---|---|---|
| **A 滑动窗口** | 只把最近 6 条对话塞给 DeepSeek | 没有记忆系统的普通 Agent 能答对多少？（远期记忆直接丢） |
| **B 全上下文** | 把全部对话塞给 DeepSeek | 不建记忆、暴力拉满上下文的天花板和成本 |
| **C 朴素 RAG** | 向量检索固定 Top-3 + DeepSeek | 把 Jev-Mem 的**决策面卸掉**、只留向量库，损失多少？（这是"Jev 决策面价值"的消融） |
| **D Jev-Mem** | 完整管线：Jev 决策（或兜底）+ 图检索 + DeepSeek | 加上 System-One 控制后的效果 |

**评测协议**：DeepSeek 做 LLM-as-a-Judge（0.0–1.0 语义分，复用仓库的评分量表），温度 0；每个臂每题记录答案、端到端延迟（含检索+生成）、prompt token 数。全部真跑实测，没有手填数字。

**诚实声明**：这是 16 轮对话的玩具实验，单次运行、有噪声，**用于理解机制差异，不构成对论文 LoCoMo 数字的复现**。

In [ ]:
CONVERSATION = DATASET["observations"]
QUESTIONS = DATASET["questions"]
print(f"对话 {len(CONVERSATION)} 轮；问题 {len(QUESTIONS)} 个")


In [ ]:
# A/B/C/D 使用同一套答题提示词、相同角色和时间戳；
# 使用量取自每一次响应，答案与 judge 分开记录。
from Jev_mem.pipeline.experiment import run_experiment


In [ ]:
RUN = session.run(DATASET, OUTPUT)
SUMMARY = json.loads((RUN / "summary.json").read_text())
RESULTS = [json.loads(line) for line in (RUN / "evaluations.jsonl").read_text().splitlines()]
print("Run:", RUN)
print("Metric:", SUMMARY["score_metric"])


In [ ]:
for arm, values in SUMMARY["arms"].items():
    print(arm, "score=", values["mean_score"], "prompt_tokens=", values["mean_prompt_tokens"],
          "coverage=", f"{values['scored']}/{values['questions']}")
print("SVG report:", RUN / "report.svg")


In [ ]:
for record in RESULTS:
    if record["question_id"] == "0":
        print(record["arm"], record["answer"], "score=", record["score"], "status=", record["status"])


### 这一轮怎么读（实测解读，数字以运行结果为准）

**1. Jev 决策面有增益：D 稳定高于 C。** 差距主要来自时间题——朴素 RAG 靠语义相似度找证据，"最早的出差计划是哪天提出来的"这类要按**时间线**找的问题，它检索不到就直接答"记忆中没有"；Jev-Mem 的路由判出"要查时间关系"，沿 temporal 图精确召回具体日期。两轮实测（本地 MiniLM、Qwen3-Embedding）C 臂时间题都是 0 分——**embedding 再强也不是时间推理的解**，这是决策面的正交价值。

**2. 换强 embedding 对两条向量检索路径都有帮助**：本地 MiniLM 换成 Qwen3-Embedding 后，C 涨约 0.06-0.08（单跳受益最直接），D 小幅跟涨——D 对 C 的领先保持稳定，说明决策面的贡献与 embedding 质量解耦。

**3. 但别迷信记忆系统：16 轮的短对话，全上下文就是天花板**（还不用检索）。论文里 Full Context 只有 0.481，是因为 LoCoMo 是几百轮的超长对话——塞不下、注意力稀释、token 爆炸。教训：**对话短就别上记忆系统；塞不下上下文了，才是 Jev-Mem 的主场。**

**4. D 臂 token 最省**——只把筛选过的证据带进 System-Two；延迟最贵——每道题要过真 Jev 决策链 + embedding 网络往返。这和论文报告的 0.93s 对基线 1.47~32.68s 是同一个量级故事：花小钱买决策，省下大模型的检索路径开销。

**5. D 臂丢的分有一半丢在"答案太短"**：仓库的 System-Two 提示词要求"5-6 个词简答"，中文多跳题就漏细节（"改到下周三，推迟了"没提"重新订票"）。这是提示词工程问题不是架构问题。对抗题四个臂都稳定满分（都不编造），但 B 为此付了最贵的 token 账单。

In [ ]:
# Offline 没有真实 API token 账单，不计算 token 节省倍数。
print("建图秒数:", SUMMARY["build_seconds"])
print("节点 / 边:", SUMMARY["nodes"], SUMMARY["edges"])
print("数据、模型配置、逐题回答、judge 使用量与图都已保存至:", RUN)
session.close()


## 7. 把玩具实验放回论文的坐标系

论文在 LoCoMo（真实超长多会话英文对话）上、用 GPT-4o-mini 做 System-Two 的完整结果：

| 方法 | 多跳 | 时间 | 开放域 | 单跳 | 对抗 | **总分** | 建记忆 (s) | 查询延迟 (s) |
|---|---|---|---|---|---|---|---|---|
| Full Context | 0.468 | 0.562 | 0.486 | 0.630 | 0.205 | 0.481 | — | 1.74 |
| A-MEM | 0.495 | 0.474 | 0.385 | 0.653 | 0.616 | 0.580 | 3,636 | 2.26 |
| MemoryOS | 0.552 | 0.422 | 0.504 | 0.674 | 0.428 | 0.553 | 3,276 | 32.68 |
| Nemori | 0.569 | 0.649 | 0.485 | 0.764 | 0.325 | 0.590 | 1,044 | 2.59 |
| MAGMA | 0.528 | **0.650** | 0.517 | 0.776 | 0.742 | 0.700 | 1,404 | 1.47 |
| **Jev-Mem** | **0.623** | 0.637 | **0.618** | **0.802** | **0.962** | **0.777** | **158** | **0.93** |

论文数字里最值得注意的三点：

1. **对抗题 0.962**（最强基线 0.742）——"记忆里没有就说没有"。全上下文基线只有 0.205：把一坨原文塞给模型，它反而更爱编。Jev-Mem 的 stopping 决策链里专门有一条"缺什么证据/有矛盾吗"，检索层面就知道"证据不在这"，System-Two 拿到的是干净证据集；
2. **建记忆 158 秒 vs 别人的 1000–3600 秒**——把"每条记忆一次大模型自由生成"换成"一批类型化小决策"，这是数量级的差别，也是 Jev 这种专用小模型存在的意义；
3. **多跳 +0.094、开放域 +0.101**——自适应路由和按需扩展图视图，换来了固定 Top-K 检索拿不到的证据组合。

玩具实验（第 6 节）的**实测解读见上一节"这一轮怎么读"**：D 臂对 C 臂的增益、token 对 B 臂的优势都实测成立，但短对话里全上下文的天花板优势同样真实。数量级结论（6.6×、11%）只能在 LoCoMo 全集上检验，仓库提供了 `python -m jev_mem.benchmarks.locomo` 入口，需要下载 LoCoMo 数据集（见 `data/README.md`）。

## 8. 意义与价值

**对 Agent 工程的三个直接意义**

1. **记忆操作的成本结构被改写了。** 此前每条记忆的写入、每次检索的中间判断，都是一次"按 token 计费的大模型税"，而且站在主对话循环的关键路径上。Jev-Mem 把这些操作变成高频、批量、类型化的小决策（本机实测：16 条观察建图，代码兜底路径 0.2 秒、真 Jev 决策 9.7 秒——每条约 0.6 秒，对照论文里大模型记忆系统动辄上千秒），大模型调用只剩"最终答题"一次。对要长期跑、频繁读写的个人 Agent 和企业 Agent，这是从量级上省钱的架构。
2. **记忆系统第一次变得"可审计、可调试"。** 每个决策是带概率的结构化输出，全量落在 `decisions.jsonl` 里：为什么没存、为什么走这条边、为什么停，都能回放。传统记忆系统的自由生成中间过程是黑箱，出了坏案例只能重试。
3. **"代码拥有流程，模型提供常识"的范式落地。** 预算分配用最大余数法、时间先后关系由代码直接建、关系边类型是枚举、上限全是显式配置——模型只回答代码无法查表的那部分语义判断。决策面还可整体替换：本文第 4 节用 50 行 mock 就顶上了 Jev 的位置，管线一行不改。

**对 Jev 生态的意义**

- Jev-Mem 是目前最完整的"Jev 作为 System-One 控制面"的参考实现：7 组问题、26 个类型化决策点，覆盖了一条记忆管线的全部语义判断。想给自己的场景设计决策面，这是最好的模板（对照 `memory/jev_questions.py` 读，每个问题的指令怎么写、state 怎么组织，都有讲究）。
- 它也验证了 Jev 定价模型（输入计费、输出免费、低延迟）在"高频小决策"场景下的商业合理性：决策调用量大，但每次都极小。

**诚实的局限**

- 论文结果基于英文 LoCoMo + GPT-4o-mini + 全量评测脚本；本 notebook 的中文玩具实验只验证机制和方向，不构成效果背书；
- 中文场景有两个工程细节要注意：仓库默认的 `all-MiniLM-L6-v2` 是英文 embedding（本 notebook 已换成多语言模型），BM25 关键词通道对中文分词不友好（主通道是向量锚点，影响有限但存在）；
- 无 Typesafe key 时决策走确定性兜底，质量必然打折——真实 Jev 的决策质量是这套架构的成立前提，值得单独实验（对照 cookbook 第 15 节 jevbench）；
- 记忆默认"全保留 + 四视图建边"，图规模随对话轮数线性增长，超长历史的图维护成本论文没有给出长期曲线。

## 9. 接着可以玩什么

| 想做的事 | 入口 |
|---|---|
| 打开准入闸门（该不该存也交给 Jev 判） | `JevMemConfig(admission_enabled=True)` |
| 真 LoCoMo 子集复现 | `python -m jev_mem.benchmarks.locomo --dataset data/locomo10.json --sample 0 --max-questions 10` |
| Jev 决策消融（读或写单独卸掉） | CLI `--no-jev-write` / `--no-jev-read` |
| 换 System-Two 对照（GPT-4o-mini vs DeepSeek） | `ANSWER_MODEL` + `OPENAI_BASE_URL` 环境变量 |
| 更换实验配置（更宽的锚点/预算） | `--jev-config config/jev_mem_accuracy.json` |
| LongMemEval 基准 | `python -m jev_mem.benchmarks.longmemeval` |

**引用**

```bibtex
@software{jev_mem,
  title = {{Jev-Mem}: System-One Controlled Agentic Memory},
  author = {Jiang, Dongming and Li, Yi and Li, Bingzhe},
  year = {2026},
  url = {https://github.com/libingzheren/Jev-Mem},
  note = {arXiv:2609.23986}
}
```

公众号解读见 [智猩猩AI《Agent记忆有了Jev新方案，登上Hugging Face趋势榜！》](https://mp.weixin.qq.com/s/uJr6Bb8OrNMDySXs3m7CpQ)（2026-09-21）。